# Model Visualization Notebook
This notebook evaluates models and visualizes qualitative differences.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from tqdm import tqdm

def compute_f1(pred, label, num_classes=5):
    pred = pred.flatten()
    label = label.flatten()
    return f1_score(label, pred, average='macro')


In [ ]:
from tqdm import tqdm

def evaluate_model(model, dataloader, featured, need_argmax, device='cuda', show_inner=False):
    model.eval()
    results = []

    total_batches = len(dataloader)

    with torch.no_grad():
        # 外层进度条（核心）
        for fimgs, labels, imgs in tqdm(dataloader, total=total_batches, desc="Evaluating", ncols=100):

            if featured:
                imgs = fimgs.to(device)
            else:
                imgs = imgs.to(device)

            labels = labels.to(device)

            outputs = model(imgs)

            if need_argmax:
                preds = torch.argmax(outputs, dim=1)
            else:
                preds = outputs.squeeze(1)

            batch_size = imgs.size(0)

            # 内层进度条（可选）
            iterator = range(batch_size)
            if show_inner:
                iterator = tqdm(iterator, leave=False, desc="Batch items", ncols=80)

            for i in iterator:
                f1 = compute_f1(
                    preds[i].cpu().numpy(),
                    labels[i].cpu().numpy()
                )

                results.append({
                    'image': imgs[i].cpu(),
                    'label': labels[i].cpu(),
                    'pred': preds[i].cpu(),
                    'f1': round(f1, 2)
                })

    return results

In [ ]:
def select_best_samples(all_model_results, main_model_idx=0, top_k=4):
    selected = []

    num_samples = len(all_model_results[0])

    for i in tqdm(range(num_samples), desc="Selecting Samples"):
        main_f1 = all_model_results[main_model_idx][i]['f1']
        others_f1 = [
            all_model_results[j][i]['f1']
            for j in range(len(all_model_results)) if j != main_model_idx
        ]

        score = (main_f1, -np.mean(others_f1))
        selected.append((i, score))

    selected = sorted(selected, key=lambda x: (x[1][0], x[1][1]), reverse=True)

    return [idx for idx, _ in selected[:top_k]]

In [ ]:
def visualize(models_results, indices, class_colors=None):
    num_models = len(models_results)
    rows = len(indices)
    cols = 2 + num_models

    plt.figure(figsize=(4 * cols, 4 * rows))

    for r, idx in enumerate(indices):
        sample = models_results[0][idx]

        # Original image
        plt.subplot(rows, cols, r * cols + 1)
        img = sample['image'].permute(1, 2, 0).numpy()
        plt.imshow(img)
        plt.title('Image')
        plt.axis('off')

        # Label
        plt.subplot(rows, cols, r * cols + 2)
        plt.imshow(sample['label'], cmap='jet')
        plt.title('Label')
        plt.axis('off')

        # Predictions
        for m in range(num_models):
            plt.subplot(rows, cols, r * cols + 3 + m)
            pred = models_results[m][idx]['pred']
            f1 = models_results[m][idx]['f1']
            plt.imshow(pred, cmap='jet')
            plt.title(f'Model {m+1} | F1={f1}')
            plt.axis('off')

    plt.tight_layout()
    plt.show()


In [ ]:
try:
    import kagglehub
except ImportError:
    !pip install kagglehub

import torch

from src.benchmark.ModelBenchmark import evaluate_model_on_dataset
from src.dataset.DroneSegDataSet import MyDataset
from src.dataset.CheckDataset import check_dataset
from torchvision import transforms

# 加载数据集
ds_path = check_dataset()
dataset = MyDataset(
    'drone_seg_dataset/classes_dataset/classes_dataset/original_images',
    'drone_seg_dataset/classes_dataset/classes_dataset/label_images_semantic',
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]),
    # data_enforcement=True,
)

n_classes = 5
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

n_indices = 170
random_indices = np.random.choice(len(dataset), n_indices, replace=False)
selected_indices = [
    random_indices[i] for i in range(n_indices)
        if dataset.__getitem__(random_indices[i])[1].unique().size(0) > 3
]

dataloader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(dataset, selected_indices),
    batch_size=2,
    shuffle=False,
    num_workers=2,
)


In [ ]:
# 测试 U-Net 基线模型

from src.model.baseline_model.Unet_model import UNet
model1 = UNet(in_channels=3, num_classes=n_classes).to(device)

param_path = 'resources/para/baseline_unet.pth'
model1.load_state_dict(torch.load(param_path, map_location=device))

result1 = evaluate_model(
    model=model1,
    dataloader=dataloader,
    featured=False,
    need_argmax=True,
)

del model1


In [ ]:
# 测试自创模型 1

from src.model.Model1.DroneSegModel import DroneSegModel
model2 = DroneSegModel(
    max_channels=64,
).to(device)

param_path = 'resources/para/model1.pth'
model2.load_state_dict(torch.load(param_path, map_location=device))

result2 = evaluate_model(
    model=model2,
    dataloader=dataloader,
    featured=True,
    need_argmax=True,
)

del model2


In [ ]:
# 测试自创模型 2

from src.model.Model2.DroneSegModel import DroneSegModel
model3 = DroneSegModel().to(device)

param_path = 'resources/para/model2.pth'
model3.load_state_dict(torch.load(param_path, map_location=device))

result3 = evaluate_model(
    model=model3,
    dataloader=dataloader,
    featured=True,
    need_argmax=True,
)

del model3


In [ ]:
# 测试 U-Net 多分枝模型，合并方法使用优先级排列（permutation）

from src.model.MultiU_NetModel import MultiU_Net
model4 = MultiU_Net(
    in_channel=22,
    depth=[3] * 5,
    depthwise_separable=False,
    combine_method='permutation',
).to(device)

import os
para_dir = 'resources/para'
para_list = [
    os.path.join(para_dir, 'unet_branch0.pth'),
    os.path.join(para_dir, 'unet_branch1.pth'),
    os.path.join(para_dir, 'unet_branch2.pth'),
    os.path.join(para_dir, 'unet_branch3.pth'),
    os.path.join(para_dir, 'unet_branch4.pth'),
]
model4.read_param(para_list)

result4 = evaluate_model(
    model=model4,
    dataloader=dataloader,
    featured=True,
    need_argmax=False,
)

del model4


In [ ]:
# 测试 U-Net 多分枝模型，合并方法使用 1*1 卷积层

from src.model.MultiU_NetModel import MultiU_Net
model5 = MultiU_Net(
    in_channel=22,
    depth=[3] * 5,
    depthwise_separable=False,
    combine_method='out_layer',
).to(device)

import os
para_dir = 'resources/para'
para_list = [
    os.path.join(para_dir, 'unet_branch0.pth'),
    os.path.join(para_dir, 'unet_branch1.pth'),
    os.path.join(para_dir, 'unet_branch2.pth'),
    os.path.join(para_dir, 'unet_branch3.pth'),
    os.path.join(para_dir, 'unet_branch4.pth'),
]
out_para = os.path.join(para_dir, 'out_conv.pth')
model5.read_param(para_list, out_para)

result5 = evaluate_model(
    model=model5,
    dataloader=dataloader,
    featured=True,
    need_argmax=True,
)

del model5


In [ ]:
all_results = [result1, result2, result3, result4, result5]

indices = select_best_samples(all_results, main_model_idx=0, top_k=4)

visualize(all_results, indices)